# Notebook 1: FRIP Signal Generation (GEE)

This notebook computes the raw Flooding Role in Productivity (FRIP) signal using Google Earth Engine.
It follows a strict modular structure:
1. Setup & Configuration
2. Methodological Logic (Functions)
3. Unit Tests
4. Execution (Asset Export)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee
import geemap

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")

# Output destination
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study Regions
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
STUDY_REGION = ee.FeatureCollection([
    ee.Feature(CONGO_BBOX, {'basin': 'Congo'}),
    ee.Feature(AMAZON_BBOX, {'basin': 'Amazon'})
])

# Scales (meters)
SCALES = list(range(5000, 105000, 5000))

# Datasets
FOREST_MASK = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10
FOREST_COVER_THRESHOLD = 0.95
MODIS_SCALE = 500  # Native MODIS NPP resolution in meters
MERIT_HYDRO = 'MERIT/Hydro/v1_0_1'
GLOFAS = 'JRC/CEMS_GLOFAS/FloodHazard/v2_1'
MODIS_NPP = 'MODIS/061/MOD17A3HGF'
YEARS = list(range(2001, 2024))

print("✓ Configuration loaded.")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

def load_and_mask_base_layers():
    """Loads JRC TMF and MERIT Hydro layers. Builds a unified 500m pristine forest mask.
    
    The 30m JRC TMF binary forest mask is aggregated to MODIS 500m resolution.
    Only MODIS pixels where >=95% of underlying 30m pixels are intact forest
    are retained. This ensures NPP values are not contaminated by non-forest
    land cover within the coarse pixel.
    """
    # Load 30m JRC TMF
    tmf_col = ee.ImageCollection(FOREST_MASK)
    tmf = tmf_col.mosaic().setDefaultProjection(tmf_col.first().projection())
    forest_mask_30m = tmf.eq(FOREST_CLASS)
    
    # MERIT Hydro connectivity mask
    merit = ee.Image(MERIT_HYDRO)
    hnd_mask = merit.select('hnd').gt(0)
    
    # Aggregate 30m forest mask to 500m MODIS resolution (~277 pixels, under 65535 limit)
    modis_proj = ee.Projection('EPSG:4326').atScale(MODIS_SCALE)
    forest_fraction_500m = forest_mask_30m.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).reproject(crs=modis_proj)
    
    # Pristine forest mask: only 500m pixels with >=95% intact forest cover
    pristine_mask = forest_fraction_500m.gte(FOREST_COVER_THRESHOLD)
    
    # Combined mask: pristine forest AND hydrologically connected
    combined_mask = pristine_mask.updateMask(hnd_mask)
    return combined_mask

def load_frip_inputs(combined_mask):
    """Loads MODIS NPP and GLOFAS Flood Depth, applying the combined mask."""
    glofas = ee.ImageCollection(GLOFAS)
    depth_bands = ['RP10_depth', 'RP20_depth', 'RP50_depth', 'RP75_depth', 'RP100_depth', 'RP200_depth', 'RP500_depth']
    flood_depth = glofas.mosaic().select(depth_bands).reduce(ee.Reducer.sum()).rename('depth').updateMask(combined_mask)
    
    modis = ee.ImageCollection(MODIS_NPP).select('Npp')
    def get_annual(year):
        img = modis.filter(ee.Filter.calendarRange(year, year, 'year')).first()
        return img.updateMask(combined_mask).set('year', year)
    
    annual_npp = ee.ImageCollection.fromImages(ee.List(YEARS).map(get_annual))
    mean_npp = annual_npp.mean()
    npp_proj = annual_npp.first().projection()
    
    return flood_depth, annual_npp, mean_npp, npp_proj

def compute_spatial_correlation(npp_img, flood_depth, scale, npp_proj):
    """Computes spatial Spearman correlation between NPP and Flood Depth within a grid cell.
    
    Uses setDefaultProjection at the target scale so reduceResolution knows the
    output pixel size, but does NOT call reproject() — the final projection is
    handled by the Export task. This avoids the 'Reprojection output too large' error.
    """
    stack = ee.Image.cat([npp_img.rename('npp'), flood_depth.rename('depth')]).setDefaultProjection(npp_proj)
    
    # Set the target coarse projection for reduceResolution
    target_proj = npp_proj.atScale(scale)
    
    spearman = stack.setDefaultProjection(npp_proj).reduceResolution(
        reducer=ee.Reducer.spearmansCorrelation(),
        maxPixels=65535
    ).setDefaultProjection(target_proj)
    
    return spearman.select('correlation')

def build_frip_assets(scale):
    """Orchestrates the computation for a given scale, returning cross-sectional and annual FRIP images."""
    combined_mask = load_and_mask_base_layers()
    flood_depth, annual_npp, mean_npp, npp_proj = load_frip_inputs(combined_mask)
    
    # Cross-sectional FRIP
    frip_cross = compute_spatial_correlation(mean_npp, flood_depth, scale, npp_proj).rename(f'FRIP_{scale}')
    
    # Annual FRIP
    def compute_annual(img):
        year = ee.Number(img.get('year')).format('%04d')
        corr = compute_spatial_correlation(img, flood_depth, scale, npp_proj)
        return corr.rename(ee.String('FRIP_').cat(year))
    
    frip_annual_col = annual_npp.map(compute_annual)
    frip_annual_img = frip_annual_col.toBands()
    
    # Clean up band names for annual image (remove the list index prefix)
    band_names = [f'FRIP_{y}' for y in YEARS]
    frip_annual_img = frip_annual_img.rename(band_names)
    
    return frip_cross, frip_annual_img, npp_proj

print("\u2713 Methodological functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running Unit Tests for FRIP generation...")
    test_scale = 50000
    passed = 0
    failed = 0
    
    try:
        # --- Test 1: Pristine forest mask ---
        print("  [1/6] Testing pristine forest mask construction...")
        combined_mask = load_and_mask_base_layers()
        assert isinstance(combined_mask, ee.Image), "Mask is not an ee.Image"
        passed += 1
        print("    ✓ Mask constructed successfully")
        
        # --- Test 2: FRIP input loading ---
        print("  [2/6] Testing FRIP input data loading...")
        flood_depth, annual_npp, mean_npp, npp_proj = load_frip_inputs(combined_mask)
        assert isinstance(flood_depth, ee.Image), "Flood depth is not an ee.Image"
        assert isinstance(mean_npp, ee.Image), "Mean NPP is not an ee.Image"
        
        npp_bands = mean_npp.bandNames().getInfo()
        assert len(npp_bands) == 1, f"Mean NPP should have 1 band, got {len(npp_bands)}"
        
        annual_count = annual_npp.size().getInfo()
        assert annual_count == len(YEARS), f"Expected {len(YEARS)} annual images, got {annual_count}"
        passed += 1
        print(f"    ✓ Loaded {annual_count} annual NPP images + flood depth")
        
        # --- Test 3: Build full FRIP assets ---
        print("  [3/6] Testing FRIP asset construction (cross-sectional + annual)...")
        frip_cross, frip_annual, proj = build_frip_assets(test_scale)
        assert isinstance(frip_cross, ee.Image), "Cross-sectional output is not an ee.Image"
        assert isinstance(frip_annual, ee.Image), "Annual output is not an ee.Image"
        passed += 1
        print("    ✓ FRIP assets built")
        
        # --- Test 4: Band name validation ---
        print("  [4/6] Validating band names...")
        cross_bands = frip_cross.bandNames().getInfo()
        assert len(cross_bands) == 1, f"Cross-sectional should have 1 band, got {len(cross_bands)}"
        assert cross_bands[0] == f'FRIP_{test_scale}', f"Expected FRIP_{test_scale}, got {cross_bands[0]}"
        
        annual_bands = frip_annual.bandNames().getInfo()
        assert len(annual_bands) == len(YEARS), f"Expected {len(YEARS)} annual bands, got {len(annual_bands)}"
        assert annual_bands[0] == 'FRIP_2001', f"First band should be FRIP_2001, got {annual_bands[0]}"
        assert annual_bands[-1] == 'FRIP_2023', f"Last band should be FRIP_2023, got {annual_bands[-1]}"
        passed += 1
        print(f"    ✓ Cross-sectional: {cross_bands}")
        print(f"    ✓ Annual: {annual_bands[0]} ... {annual_bands[-1]} ({len(annual_bands)} bands)")
        
        # --- Test 5: Deep execution (server-side evaluation) ---
        print("  [5/6] Deep execution test (forces GEE to evaluate graph over Congo test region)...")
        tiny_test_region = ee.Geometry.Rectangle([15.0, 0.0, 15.5, 0.5])
        test_val = frip_cross.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=tiny_test_region,
            scale=test_scale,
            maxPixels=1e9
        ).getInfo()
        
        assert isinstance(test_val, dict), "Reduction did not return a dictionary"
        band_key = f'FRIP_{test_scale}'
        assert band_key in test_val, f"Missing {band_key} in result"
        frip_value = test_val[band_key]
        
        # Sanity check: Spearman r must be in [-1, 1]
        if frip_value is not None:
            assert -1 <= frip_value <= 1, f"FRIP value {frip_value} outside [-1, 1] range"
        passed += 1
        print(f"    ✓ FRIP value at test region: {frip_value}")
        
        # --- Test 6: Annual deep execution ---
        print("  [6/6] Deep execution test on annual FRIP (first year)...")
        annual_val = frip_annual.select('FRIP_2001').reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=tiny_test_region,
            scale=test_scale,
            maxPixels=1e9
        ).getInfo()
        
        assert 'FRIP_2001' in annual_val, "Missing FRIP_2001 in annual result"
        annual_frip = annual_val['FRIP_2001']
        if annual_frip is not None:
            assert -1 <= annual_frip <= 1, f"Annual FRIP value {annual_frip} outside [-1, 1] range"
        passed += 1
        print(f"    ✓ Annual FRIP_2001 value at test region: {annual_frip}")
        
        print(f"\n{'='*50}")
        print(f"  ✓ ALL {passed} TESTS PASSED")
        print(f"{'='*50}")
        
    except AssertionError as e:
        failed += 1
        print(f"\n  ✗ Test Failed ({passed} passed, {failed} failed): {e}")
    except Exception as e:
        failed += 1
        print(f"\n  ✗ Unexpected Error ({passed} passed, {failed} failed): {e}")

# Execute tests
run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXECUTION (ASSET EXPORT)
# =============================================================================

def safe_start(task, asset_id):
    """Deletes existing asset if present, then starts the export task."""
    try:
        ee.data.deleteAsset(asset_id)
        print(f"  Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_all_scales(dry_run=True):
    tasks = []
    print(f"Configuring export tasks for {len(SCALES)} scales...")
    
    for scale in SCALES:
        frip_cross, frip_annual, proj = build_frip_assets(scale)
        
        cross_id = f'{ASSET_ROOT}/FRIP_raw/FRIP_{scale}'
        task_cross = ee.batch.Export.image.toAsset(
            image=frip_cross,
            description=f'FRIP_{scale}_Export',
            assetId=cross_id,
            region=STUDY_REGION.geometry(),
            scale=scale,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task_cross, cross_id))
        
        ann_id = f'{ASSET_ROOT}/FRIP_raw/FRIP_Annual_{scale}'
        task_ann = ee.batch.Export.image.toAsset(
            image=frip_annual,
            description=f'FRIP_Annual_{scale}_Export',
            assetId=ann_id,
            region=STUDY_REGION.geometry(),
            scale=scale,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        tasks.append((task_ann, ann_id))
    
    print(f"\u2713 Configured {len(tasks)} export tasks.")
    if dry_run:
        print("DRY RUN: Tasks created but not started. Call export_all_scales(dry_run=False) to begin.")
    else:
        for task, asset_id in tasks:
            safe_start(task, asset_id)
        print(f"\u2713 {len(tasks)} tasks started! Monitor at https://code.earthengine.google.com/tasks")

# To execute the exports, set dry_run=False
export_all_scales(dry_run=True)